# Particle Physics Generators and GPU-Friendly Analysis

Checked against public upstream project status in May 2026.

This notebook is a practical introduction to three widely used HEP generator stages:
- MadGraph5_aMC@NLO for hard-scattering matrix elements
- PYTHIA 8 for parton showers, hadronization, and decays
- EvtGen for heavy-flavor decay modeling

Important scope note: these upstream tools are still primarily CPU-oriented C++ or command-line workflows. This notebook does **not** pretend that MadGraph, PYTHIA, or EvtGen are native GPU Python packages. Instead, it shows where GPU acceleration helps today: parallel matrix-element studies, large-array analysis, and toy decay kernels that mirror generator workloads.

## Learning Objectives
- Understand what each generator stage is responsible for
- Distinguish upstream-supported workflows from notebook-friendly approximations
- Compare CPU and GPU performance for parallel physics calculations
- Practice analysis patterns that scale to real generated event samples

We use three **separate** examples in this lesson:
1. A leading-order e+e- -> mu+mu- matrix-element study
2. A PYTHIA-inspired radiation analysis example
3. An EvtGen-style B -> J/psi K decay study

These examples are pedagogical building blocks, not one single end-to-end production chain.

---
## 1. Environment and Upstream Tool Checks

We start by checking the local Python stack and looking for the external generator tools.

Upstream status to keep in mind:
- MadGraph5_aMC@NLO is normally used as a command-line application that writes parton-level events, often in LHE format.
- PYTHIA 8.317 is currently distributed as a C++ package with examples and manuals on the official site.
- EvtGen is typically built as a standalone C++ package or inside experiment software stacks; no maintained pure-Python workflow is assumed here.

The notebook itself only requires NumPy and Matplotlib. CuPy is optional for GPU acceleration.

In [ ]:
import importlib.util
import os
import platform
import shutil
import sys


def module_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


mandatory_modules = ["numpy", "matplotlib"]
optional_modules = {
    "cupy": "Optional GPU array backend",
}

missing_mandatory = [name for name in mandatory_modules if not module_available(name)]

print("Python environment")
print(f"- Python: {sys.version.split()[0]}")
print(f"- Platform: {platform.platform()}")

if missing_mandatory:
    print("\nMissing required Python modules:")
    for name in missing_mandatory:
        print(f"- {name}")
    raise RuntimeError(
        "Install the missing modules before running the rest of the notebook. "
        "A minimal setup is: python -m pip install numpy matplotlib"
    )

print("\nRequired Python modules")
for name in mandatory_modules:
    print(f"- {name}: available")

print("\nOptional Python modules")
for name, description in optional_modules.items():
    status = "available" if module_available(name) else "not installed"
    print(f"- {name}: {status} ({description})")

tool_checks = {
    "MadGraph5_aMC@NLO CLI": shutil.which("mg5_aMC") or shutil.which("mg5"),
    "PYTHIA 8 helper": shutil.which("pythia8-config"),
    "EvtGen hint": shutil.which("evtgen") or os.environ.get("EVTGEN_ROOT"),
}

print("\nExternal generator tools")
for label, value in tool_checks.items():
    if value:
        print(f"- {label}: found ({value})")
    else:
        print(f"- {label}: not found in this environment")

print("\nThis notebook will still run without the external generators.")
print("When the external tools are missing, we use accurate toy studies rather than pretending to call unsupported Python APIs.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings

warnings.filterwarnings("ignore")

GEV2_TO_PB = 0.389379338e9

gpu_available = False
cp = None
xp = np

try:
    import cupy as cp  # type: ignore
    test_array = cp.arange(8, dtype=cp.float32)
    _ = cp.sum(test_array)
    gpu_available = True
    xp = cp
    print("GPU backend: CuPy available")
    print(f"- Device: {cp.cuda.Device().name}")
except Exception as error:
    print(f"GPU backend: unavailable ({str(error)[:80]})")
    print("- Falling back to NumPy on CPU")

rng = np.random.default_rng(42)
print(f"\nArray backend for mandatory cells: {'CuPy' if gpu_available else 'NumPy'}")
print(f"Unit conversion used below: 1 GeV^-2 = {GEV2_TO_PB:.6e} pb")

In [ ]:
gpu_plan = [
    ("Environment checks", False, "CPU", "Package imports and executable discovery are CPU-side"),
    ("MadGraph-style matrix-element kernel", gpu_available, "GPU if available", "Parallel phase-space evaluation is a good GPU workload"),
    ("PYTHIA-style event generation context", False, "CPU", "The real generator is not being run on the GPU here"),
    ("PYTHIA-style array analysis", gpu_available, "GPU if available", "Large-array reductions and derived observables benefit from GPU execution"),
    ("EvtGen-style decay kernel", gpu_available, "GPU if available", "Independent toy decay calculations are parallel over events"),
    ("Summary plotting", False, "CPU", "Matplotlib rendering is CPU-side"),
    ("Cross-section Monte Carlo solution", gpu_available, "GPU if available", "Sampling and reductions are data-parallel"),
]

print("GPU execution map")
for label, enabled, mode, note in gpu_plan:
    status = "active" if enabled else "not active"
    print(f"- {label}: {mode} ({status})")
    print(f"  {note}")

---
## 1.1 GPU Execution Map

The notebook contains a mix of CPU-only generator context and GPU-capable numerical kernels.

### What can actually run on the GPU here
- Cell 8: the MadGraph-style matrix-element study can run on the GPU when CuPy is available, because it is a large parallel array calculation.
- Cell 11: the PYTHIA-style analysis can run on the GPU, but only for **post-processing** of arrays that were already produced on the CPU.
- Cell 14: the EvtGen-style decay kernel can run on the GPU because the toy decay calculations are parallel over many events.
- Cell 20: the Monte Carlo cross-section solution can run on the GPU because the sampling and reduction are data-parallel.

### What does not run on the GPU here
- The MadGraph5_aMC@NLO executable itself does not run on the GPU in this notebook.
- The upstream PYTHIA 8 generator itself does not run on the GPU in this notebook.
- The upstream EvtGen package itself does not run on the GPU in this notebook.
- Package checks, plotting, notebook control flow, and most Python orchestration remain CPU-side.

### Where GPU acceleration is most useful
- Repeating the same arithmetic over many independent events or phase-space points.
- Histogram preparation, reductions, and derived-variable calculations on large arrays.
- Monte Carlo integration with large sample counts.

### Where GPU acceleration is usually less useful
- Small event samples where GPU launch and transfer overhead dominate.
- External command-line tools that are fundamentally CPU-oriented.
- Plot rendering and lightweight notebook housekeeping.

---
## 2. Physics Background

### The Generator Stages We Care About

A modern event-generation workflow usually separates the physics into stages:

1. **Hard process**: matrix elements for the short-distance scattering
2. **Shower and hadronization**: additional QED/QCD radiation and nonperturbative final-state modeling
3. **Particle decays**: specialized decay models, especially for heavy flavor

For this notebook, we keep those stages conceptually separate so the examples stay physically meaningful:

- The MadGraph-style section studies the leading-order process e+e- -> mu+mu-.
- The PYTHIA-style section studies radiation-sensitive observables for that lepton final state.
- The EvtGen-style section switches to a separate heavy-flavor decay example, B -> J/psi K.

That separation matters. A muon-pair final state does **not** hadronize into B mesons, so we do not present it as one continuous chain.

---
## 3. MadGraph5_aMC@NLO-Style Example: Leading-Order Matrix Elements

### Upstream role
MadGraph5_aMC@NLO generates hard-scattering matrix elements and parton-level events. In real workflows you usually interact with it through its command-line interface, for example by defining a process and exporting events in LHE format.

### GPU status in this notebook
- The **upstream MadGraph program** is not running on the GPU here.
- The **numerical kernel in the next GPU-capable cell** can run on the GPU if CuPy is available.

In this notebook we mirror one piece of the hard-process workload directly: the leading-order angular dependence for e+e- -> mu+mu-. This is a good GPU teaching example because the calculation is embarrassingly parallel over many phase-space points.

For pure QED exchange, the tree-level angular dependence is proportional to 1 + cos^2(theta).

In [ ]:
def madgraph_style_cpu_simulation(nevents=10000, beam_energy=500.0):
    """
    CPU study of the LO e+e- -> mu+mu- angular distribution.

    We sample phase-space points and evaluate the QED differential cross section
    dσ/dΩ = α^2 / (4 s) * (1 + cos^2 θ)
    with s = (2 E_beam)^2.
    """
    print(f"MadGraph-style CPU study: e+e- -> mu+mu- with {nevents:,} samples")

    alpha_em = 1.0 / 137.0
    s = (2.0 * beam_energy) ** 2

    start_time = time.perf_counter()

    cos_theta = rng.uniform(-1.0, 1.0, nevents)
    phi = rng.uniform(0.0, 2.0 * np.pi, nevents)

    sin_theta = np.sqrt(1.0 - cos_theta**2)
    p_muon = beam_energy
    px_mum = p_muon * sin_theta * np.cos(phi)
    py_mum = p_muon * sin_theta * np.sin(phi)
    pz_mum = p_muon * cos_theta

    matrix_element_shape = 1.0 + cos_theta**2
    dsigma_domega = (alpha_em**2 / (4.0 * s)) * matrix_element_shape
    sigma_total_pb = 4.0 * np.pi * np.mean(dsigma_domega) * GEV2_TO_PB
    sigma_analytic_pb = (4.0 * np.pi * alpha_em**2 / (3.0 * s)) * GEV2_TO_PB

    cpu_time_ms = (time.perf_counter() - start_time) * 1000.0

    results = {
        "cos_theta": cos_theta,
        "matrix_element_shape": matrix_element_shape,
        "dsigma_domega": dsigma_domega,
        "sigma_total_pb": sigma_total_pb,
        "sigma_analytic_pb": sigma_analytic_pb,
        "muon_momenta": np.column_stack([
            np.full(nevents, beam_energy),
            px_mum,
            py_mum,
            pz_mum,
        ]),
        "computation_time": cpu_time_ms,
    }

    print(f"- CPU time: {cpu_time_ms:.2f} ms")
    print(f"- Monte Carlo sigma: {sigma_total_pb:.5f} pb")
    print(f"- Analytic sigma:    {sigma_analytic_pb:.5f} pb")
    print(f"- Relative deviation: {abs(sigma_total_pb - sigma_analytic_pb) / sigma_analytic_pb:.2%}")

    return results


mg_cpu_results = madgraph_style_cpu_simulation(nevents=10000)

In [ ]:
def madgraph_style_gpu_simulation(nevents=10000, beam_energy=500.0):
    """GPU version of the LO e+e- -> mu+mu- study."""
    if not gpu_available:
        print("GPU backend not available; skipping GPU matrix-element study")
        return None

    print(f"MadGraph-style GPU study: e+e- -> mu+mu- with {nevents:,} samples")

    alpha_em = 1.0 / 137.0
    s = (2.0 * beam_energy) ** 2

    start_event = cp.cuda.Event()
    end_event = cp.cuda.Event()
    start_event.record()

    cos_theta = cp.random.uniform(-1.0, 1.0, nevents, dtype=cp.float32)
    phi = cp.random.uniform(0.0, 2.0 * cp.pi, nevents, dtype=cp.float32)

    sin_theta = cp.sqrt(1.0 - cos_theta**2)
    p_muon = beam_energy
    px_mum = p_muon * sin_theta * cp.cos(phi)
    py_mum = p_muon * sin_theta * cp.sin(phi)
    pz_mum = p_muon * cos_theta

    matrix_element_shape = 1.0 + cos_theta**2
    dsigma_domega = (alpha_em**2 / (4.0 * s)) * matrix_element_shape
    sigma_total_pb = 4.0 * cp.pi * cp.mean(dsigma_domega) * GEV2_TO_PB
    sigma_analytic_pb = (4.0 * cp.pi * alpha_em**2 / (3.0 * s)) * GEV2_TO_PB

    end_event.record()
    end_event.synchronize()
    gpu_time_ms = cp.cuda.get_elapsed_time(start_event, end_event)

    results = {
        "cos_theta": cp.asnumpy(cos_theta),
        "matrix_element_shape": cp.asnumpy(matrix_element_shape),
        "dsigma_domega": cp.asnumpy(dsigma_domega),
        "sigma_total_pb": float(sigma_total_pb),
        "sigma_analytic_pb": float(sigma_analytic_pb),
        "muon_momenta": cp.asnumpy(cp.column_stack([
            cp.full(nevents, beam_energy, dtype=cp.float32),
            px_mum,
            py_mum,
            pz_mum,
        ])),
        "computation_time": gpu_time_ms,
    }

    print(f"- GPU time: {gpu_time_ms:.2f} ms")
    print(f"- Monte Carlo sigma: {results['sigma_total_pb']:.5f} pb")
    print(f"- Analytic sigma:    {results['sigma_analytic_pb']:.5f} pb")

    if "mg_cpu_results" in globals():
        speedup = mg_cpu_results["computation_time"] / gpu_time_ms
        print(f"- Speedup vs CPU: {speedup:.1f}x")

    return results


mg_gpu_results = madgraph_style_gpu_simulation(nevents=10000)

---
## 4. PYTHIA 8-Style Example: Radiation-Sensitive Observables

### Upstream role
PYTHIA 8 handles showers, hadronization, and many decay steps. The official upstream workflow is still centered on the C++ code base and its example programs.

### GPU status in this notebook
- The **upstream PYTHIA generator** is not running on the GPU here.
- The **analysis of large output-like arrays** can run on the GPU if CuPy is available.
- This is the most realistic GPU use case in the notebook: keep generation on CPU, accelerate bulk analysis after generation.

This notebook does **not** assume an official Python binding. Instead, it uses a radiation-inspired toy sample that captures the kind of observables you often inspect after a PYTHIA run:
- lepton transverse momentum after radiation losses
- pseudorapidity acceptance
- per-event photon multiplicity as a stand-in for ISR/FSR activity

That keeps the notebook runnable everywhere while staying honest about what upstream PYTHIA actually provides.

In [ ]:
def pythia_style_cpu_simulation(nevents=2000, beam_energy=500.0):
    """
    CPU toy model for radiation-sensitive observables after e+e- -> mu+mu-.

    This is deliberately labelled 'PYTHIA-style' rather than 'PYTHIA execution'.
    It mimics the broad effect of ISR/FSR on lepton kinematics and photon counts.
    """
    print(f"PYTHIA-style CPU study: {nevents:,} events")

    start_time = time.perf_counter()

    cos_theta = rng.uniform(-0.995, 0.995, nevents)
    born_pt = beam_energy * np.sqrt(1.0 - cos_theta**2)
    eta = np.arctanh(cos_theta)

    radiation_loss = rng.beta(2.0, 12.0, nevents)
    muon_pt = born_pt * (1.0 - 0.35 * radiation_loss)
    photon_multiplicity = rng.poisson(0.8 + 1.5 * radiation_loss)

    cpu_time_ms = (time.perf_counter() - start_time) * 1000.0

    results = {
        "muon_pt": muon_pt,
        "muon_eta": eta,
        "photon_multiplicity": photon_multiplicity,
        "radiation_loss": radiation_loss,
        "total_events": nevents,
        "computation_time": cpu_time_ms,
    }

    print(f"- CPU time: {cpu_time_ms:.2f} ms")
    print(f"- Mean muon pT: {np.mean(muon_pt):.2f} GeV")
    print(f"- Mean photon multiplicity: {np.mean(photon_multiplicity):.2f}")
    print("- Interpretation: more radiation lowers the visible muon pT and raises the photon count")

    return results


pythia_cpu_results = pythia_style_cpu_simulation(nevents=50000)

In [ ]:
def pythia_style_gpu_analysis(cpu_results):
    """
    GPU analysis of large event arrays produced by a shower study.


    This mirrors a common real-world use case: the generator itself runs on CPU,
    while histogramming and derived-variable calculations are accelerated on GPU.
    """
    if not gpu_available:
        print("GPU backend not available; skipping GPU analysis")
        return None

    print("PYTHIA-style GPU analysis of shower observables")

    start_event = cp.cuda.Event()
    end_event = cp.cuda.Event()
    start_event.record()

    muon_pt_gpu = cp.asarray(cpu_results["muon_pt"], dtype=cp.float32)
    muon_eta_gpu = cp.asarray(cpu_results["muon_eta"], dtype=cp.float32)
    photon_mult_gpu = cp.asarray(cpu_results["photon_multiplicity"], dtype=cp.float32)
    radiation_loss_gpu = cp.asarray(cpu_results["radiation_loss"], dtype=cp.float32)

    central_fraction = cp.mean(cp.abs(muon_eta_gpu) < 2.5)
    high_pt_fraction = cp.mean(muon_pt_gpu > 200.0)
    radiative_fraction = cp.mean(photon_mult_gpu > 0.0)
    mean_visible_scale = cp.mean(1.0 - 0.35 * radiation_loss_gpu)

    pt_mean = cp.mean(muon_pt_gpu)
    pt_std = cp.std(muon_pt_gpu)
    standardized = (muon_pt_gpu - pt_mean) / pt_std
    pt_skewness = cp.mean(standardized**3)
    pt_kurtosis = cp.mean(standardized**4)

    end_event.record()
    end_event.synchronize()
    gpu_time_ms = cp.cuda.get_elapsed_time(start_event, end_event)

    results = {
        "central_fraction": float(central_fraction),
        "high_pt_fraction": float(high_pt_fraction),
        "radiative_fraction": float(radiative_fraction),
        "mean_visible_scale": float(mean_visible_scale),
        "pt_moments": {
            "mean": float(pt_mean),
            "std": float(pt_std),
            "skewness": float(pt_skewness),
            "kurtosis": float(pt_kurtosis),
        },
        "computation_time": gpu_time_ms,
    }

    print(f"- GPU analysis time: {gpu_time_ms:.2f} ms")
    print(f"- Central acceptance fraction: {results['central_fraction']:.1%}")
    print(f"- High-pT fraction: {results['high_pt_fraction']:.1%}")
    print(f"- Events with photons: {results['radiative_fraction']:.1%}")

    return results


pythia_gpu_results = pythia_style_gpu_analysis(pythia_cpu_results)

---
## 5. EvtGen-Style Example: Heavy-Flavor Decays

### Upstream role
EvtGen is a specialized decay package used heavily for heavy-flavor physics. In practice it is usually built as part of a larger C++ stack or experiment framework.

### GPU status in this notebook
- The **upstream EvtGen package** is not being driven on the GPU here.
- The **toy decay kernel in the GPU-capable cell** can run on the GPU if CuPy is available.
- This kind of workload is advantageous on GPU when many independent decays are evaluated with the same arithmetic pattern.

This notebook again uses an honest approximation: a toy B -> J/psi K decay study with the right two-body kinematics, lifetime sampling, and mass smearing. That lets us discuss decay observables and parallel decay kernels without claiming direct Python control of upstream EvtGen.

In [ ]:
def evtgen_style_cpu_simulation(nevents=5000):
    """CPU toy model for B -> J/psi K decay observables."""
    print(f"EvtGen-style CPU decay study: {nevents:,} decays")

    start_time = time.perf_counter()

    m_B = 5.27963
    m_Jpsi = 3.09690
    m_K = 0.49368
    tau_B_seconds = 1.520e-12

    e_jpsi = (m_B**2 + m_Jpsi**2 - m_K**2) / (2.0 * m_B)
    e_kaon = (m_B**2 + m_K**2 - m_Jpsi**2) / (2.0 * m_B)
    p_daughter = np.sqrt(e_jpsi**2 - m_Jpsi**2)

    cos_theta = rng.uniform(-1.0, 1.0, nevents)
    phi = rng.uniform(0.0, 2.0 * np.pi, nevents)
    sin_theta = np.sqrt(1.0 - cos_theta**2)

    px_jpsi = p_daughter * sin_theta * np.cos(phi)
    py_jpsi = p_daughter * sin_theta * np.sin(phi)
    pz_jpsi = p_daughter * cos_theta

    px_kaon = -px_jpsi
    py_kaon = -py_jpsi
    pz_kaon = -pz_jpsi

    decay_times = rng.exponential(tau_B_seconds, nevents)
    jpsi_mass = m_Jpsi + rng.normal(0.0, 0.010, nevents)
    b_mass = m_B + rng.normal(0.0, 0.020, nevents)

    cpu_time_ms = (time.perf_counter() - start_time) * 1000.0

    results = {
        "Jpsi_momenta": np.column_stack([np.full(nevents, e_jpsi), px_jpsi, py_jpsi, pz_jpsi]),
        "K_momenta": np.column_stack([np.full(nevents, e_kaon), px_kaon, py_kaon, pz_kaon]),
        "decay_times": decay_times,
        "Jpsi_mass": jpsi_mass,
        "B_mass": b_mass,
        "cos_theta_decay": cos_theta,
        "computation_time": cpu_time_ms,
    }

    print(f"- CPU time: {cpu_time_ms:.2f} ms")
    print(f"- Mean lifetime: {np.mean(decay_times) * 1e12:.3f} ps")
    print(f"- J/psi mass mean: {np.mean(jpsi_mass):.4f} GeV")
    print(f"- B mass mean: {np.mean(b_mass):.4f} GeV")

    return results


evtgen_cpu_results = evtgen_style_cpu_simulation(nevents=50000)

In [ ]:
def evtgen_style_gpu_simulation(nevents=50000):
    """GPU toy model for B -> J/psi K decay observables."""
    if not gpu_available:
        print("GPU backend not available; skipping GPU decay study")
        return None

    print(f"EvtGen-style GPU decay study: {nevents:,} decays")

    m_B = cp.float32(5.27963)
    m_Jpsi = cp.float32(3.09690)
    m_K = cp.float32(0.49368)
    tau_B_seconds = cp.float32(1.520e-12)

    start_event = cp.cuda.Event()
    end_event = cp.cuda.Event()
    start_event.record()

    e_jpsi = (m_B**2 + m_Jpsi**2 - m_K**2) / (2.0 * m_B)
    e_kaon = (m_B**2 + m_K**2 - m_Jpsi**2) / (2.0 * m_B)
    p_daughter = cp.sqrt(e_jpsi**2 - m_Jpsi**2)

    cos_theta = cp.random.uniform(-1.0, 1.0, nevents, dtype=cp.float32)
    phi = cp.random.uniform(0.0, 2.0 * cp.pi, nevents, dtype=cp.float32)
    sin_theta = cp.sqrt(1.0 - cos_theta**2)

    px_jpsi = p_daughter * sin_theta * cp.cos(phi)
    py_jpsi = p_daughter * sin_theta * cp.sin(phi)
    pz_jpsi = p_daughter * cos_theta

    px_kaon = -px_jpsi
    py_kaon = -py_jpsi
    pz_kaon = -pz_jpsi

    uniform_random = cp.random.uniform(1e-7, 1.0, nevents, dtype=cp.float32)
    decay_times = -tau_B_seconds * cp.log(uniform_random)
    jpsi_mass = m_Jpsi + cp.random.normal(0.0, 0.010, nevents, dtype=cp.float32)
    b_mass = m_B + cp.random.normal(0.0, 0.020, nevents, dtype=cp.float32)

    pt_jpsi = cp.sqrt(px_jpsi**2 + py_jpsi**2)
    eta_jpsi = cp.arcsinh(pz_jpsi / cp.maximum(pt_jpsi, cp.float32(1e-6)))
    detector_efficiency = cp.mean(((pt_jpsi > 1.0) & (cp.abs(eta_jpsi) < 2.5)).astype(cp.float32))

    end_event.record()
    end_event.synchronize()
    gpu_time_ms = cp.cuda.get_elapsed_time(start_event, end_event)

    results = {
        "Jpsi_momenta": cp.asnumpy(cp.column_stack([cp.full(nevents, e_jpsi), px_jpsi, py_jpsi, pz_jpsi])),
        "K_momenta": cp.asnumpy(cp.column_stack([cp.full(nevents, e_kaon), px_kaon, py_kaon, pz_kaon])),
        "decay_times": cp.asnumpy(decay_times),
        "Jpsi_mass": cp.asnumpy(jpsi_mass),
        "B_mass": cp.asnumpy(b_mass),
        "cos_theta_decay": cp.asnumpy(cos_theta),
        "detector_efficiency": float(detector_efficiency),
        "computation_time": gpu_time_ms,
    }

    print(f"- GPU time: {gpu_time_ms:.2f} ms")
    print(f"- Mean lifetime: {float(cp.mean(decay_times)) * 1e12:.3f} ps")
    print(f"- Detector efficiency proxy: {results['detector_efficiency']:.1%}")

    if "evtgen_cpu_results" in globals():
        speedup = evtgen_cpu_results["computation_time"] / gpu_time_ms
        print(f"- Speedup vs CPU: {speedup:.1f}x")

    return results


evtgen_gpu_results = evtgen_style_gpu_simulation(nevents=50000)

---
## 6. Visualization and Performance Comparison

The plots below compare the distributions from the three studies and summarize where GPU acceleration is actually being used in this notebook:
- direct parallel evaluation for the MadGraph-style matrix-element study
- large-array analysis for the PYTHIA-style observables
- parallel decay kinematics for the EvtGen-style study

In [ ]:
def create_physics_summary_plots():
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle("Generator Studies and GPU-Friendly Analysis", fontsize=16, fontweight="bold")

    if "mg_cpu_results" in globals():
        ax = axes[0, 0]
        ax.hist(mg_cpu_results["cos_theta"], bins=50, alpha=0.7, color="steelblue", density=True, label="CPU")
        if globals().get("mg_gpu_results") is not None:
            ax.hist(
                mg_gpu_results["cos_theta"],
                bins=50,
                alpha=0.8,
                histtype="step",
                linewidth=2,
                color="darkred",
                density=True,
                label="GPU",
            )
        ax.set_xlabel("cos(theta)")
        ax.set_ylabel("Normalized entries")
        ax.set_title("MadGraph-style angular distribution")
        ax.grid(True, alpha=0.3)
        ax.legend()

        ax = axes[1, 0]
        ax.hist(mg_cpu_results["matrix_element_shape"], bins=50, alpha=0.7, color="mediumseagreen")
        ax.set_xlabel("1 + cos^2(theta)")
        ax.set_ylabel("Entries")
        ax.set_title("Matrix-element shape")
        ax.grid(True, alpha=0.3)

    if "pythia_cpu_results" in globals():
        ax = axes[0, 1]
        ax.hist(pythia_cpu_results["muon_pt"], bins=60, range=(0, 520), alpha=0.75, color="coral")
        ax.set_xlabel("Muon pT [GeV]")
        ax.set_ylabel("Entries")
        ax.set_title("PYTHIA-style visible muon pT")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

        ax = axes[1, 1]
        ax.hist(
            pythia_cpu_results["photon_multiplicity"],
            bins=np.arange(-0.5, 8.5, 1.0),
            alpha=0.75,
            color="goldenrod",
        )
        ax.set_xlabel("Photon multiplicity")
        ax.set_ylabel("Events")
        ax.set_title("Radiation activity proxy")
        ax.grid(True, alpha=0.3)

    if "evtgen_cpu_results" in globals():
        ax = axes[0, 2]
        ax.hist(evtgen_cpu_results["Jpsi_mass"], bins=60, range=(3.04, 3.15), alpha=0.75, color="mediumpurple")
        ax.axvline(3.0969, color="black", linestyle="--", linewidth=1.5, label="PDG reference")
        ax.set_xlabel("Reconstructed J/psi mass [GeV]")
        ax.set_ylabel("Entries")
        ax.set_title("EvtGen-style J/psi peak")
        ax.grid(True, alpha=0.3)
        ax.legend()

        ax = axes[1, 2]
        ax.hist(evtgen_cpu_results["decay_times"] * 1e12, bins=60, range=(0, 10), alpha=0.75, color="darkorange")
        ax.set_xlabel("Decay time [ps]")
        ax.set_ylabel("Entries")
        ax.set_title("B-hadron lifetime distribution")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\nPerformance summary")
    print("=" * 78)
    print(f"{'Study':<24} {'CPU time [ms]':<16} {'GPU time [ms]':<16} {'Comment':<20}")
    print("-" * 78)

    mg_gpu = globals().get("mg_gpu_results")
    print(
        f"{'MadGraph-style kernel':<24} "
        f"{mg_cpu_results['computation_time']:<16.2f} "
        f"{(mg_gpu['computation_time'] if mg_gpu else float('nan')):<16.2f} "
        f"{'Direct parallel kernel' if mg_gpu else 'CPU only'}"
    )

    pythia_gpu = globals().get("pythia_gpu_results")
    pythia_gpu_time = pythia_gpu["computation_time"] if pythia_gpu else float("nan")
    print(
        f"{'PYTHIA-style analysis':<24} "
        f"{pythia_cpu_results['computation_time']:<16.2f} "
        f"{pythia_gpu_time:<16.2f} "
        f"{'GPU post-processing' if pythia_gpu else 'CPU only'}"
    )

    evtgen_gpu = globals().get("evtgen_gpu_results")
    evtgen_gpu_time = evtgen_gpu["computation_time"] if evtgen_gpu else float("nan")
    print(
        f"{'EvtGen-style decay kernel':<24} "
        f"{evtgen_cpu_results['computation_time']:<16.2f} "
        f"{evtgen_gpu_time:<16.2f} "
        f"{'Direct parallel kernel' if evtgen_gpu else 'CPU only'}"
    )


create_physics_summary_plots()

---
## 7. Exercise: GPU-Accelerated Cross-Section Integration

### Problem Statement

Implement a Monte Carlo integration for the total cross section of e+e- -> mu+mu-.

Use the tree-level differential cross section

dσ/dΩ = (alpha^2 / 4 s) (1 + cos^2(theta))

with:
- alpha = 1 / 137
- s = (2 E_beam)^2

Your tasks:
1. Generate uniform samples in cos(theta) on [-1, 1]
2. Integrate over the full solid angle
3. Convert the result from GeV^-2 to pb
4. Compare CPU and GPU execution times if CuPy is available
5. Quote the statistical uncertainty

Analytical reference:

sigma_total = 4 pi alpha^2 / (3 s)

Use at least 1,000,000 samples so the timing comparison is meaningful.

In [ ]:
def exercise_cross_section_calculation():
    """Template for the exercise."""
    print("Exercise: cross-section integration for e+e- -> mu+mu-")

    alpha = 1.0 / 137.0
    e_beam = 500.0
    s = (2.0 * e_beam) ** 2
    n_samples = 1_000_000

    print(f"- Samples: {n_samples:,}")
    print(f"- CM energy: {2 * e_beam:.1f} GeV")
    print("- Fill in the CPU and GPU integrations below")

    # Suggested CPU steps:
    # 1. Sample cos_theta uniformly in [-1, 1]
    # 2. Compute dsigma_domega = (alpha**2 / (4 * s)) * (1 + cos_theta**2)
    # 3. Estimate sigma_total_pb = 4 * np.pi * np.mean(dsigma_domega) * GEV2_TO_PB
    # 4. Estimate the statistical uncertainty from the sample variance

    if gpu_available:
        print("- CuPy is available, so a GPU version can mirror the same steps")
    else:
        print("- CuPy is not available in this kernel, so only the CPU path can run here")


exercise_cross_section_calculation()

---
## 8. Exercise Solution

The solution below uses the same normalization as the MadGraph-style study above and fixes the unit conversion explicitly.

In [ ]:
def exercise_solution_cross_section():
    print("Exercise solution: GPU-friendly cross-section integration\n")

    alpha = 1.0 / 137.0
    e_beam = 500.0
    s = (2.0 * e_beam) ** 2
    n_samples = 1_000_000

    sigma_analytic_pb = (4.0 * np.pi * alpha**2 / (3.0 * s)) * GEV2_TO_PB
    print(f"Analytical result: {sigma_analytic_pb:.6f} pb")

    print("\nCPU Monte Carlo integration")
    t0 = time.perf_counter()
    cos_theta_cpu = rng.uniform(-1.0, 1.0, n_samples)
    dsigma_domega_cpu = (alpha**2 / (4.0 * s)) * (1.0 + cos_theta_cpu**2)
    sigma_cpu_pb = 4.0 * np.pi * np.mean(dsigma_domega_cpu) * GEV2_TO_PB
    sigma_cpu_unc_pb = (
        4.0
        * np.pi
        * np.std(dsigma_domega_cpu, ddof=1)
        / np.sqrt(n_samples)
        * GEV2_TO_PB
    )
    cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    print(f"- Result: {sigma_cpu_pb:.6f} ± {sigma_cpu_unc_pb:.6f} pb")
    print(f"- Time:   {cpu_time_ms:.2f} ms")
    print(f"- Pull vs theory: {(sigma_cpu_pb - sigma_analytic_pb) / sigma_cpu_unc_pb:.2f} sigma")

    if gpu_available:
        print("\nGPU Monte Carlo integration")
        start_event = cp.cuda.Event()
        end_event = cp.cuda.Event()
        start_event.record()

        cos_theta_gpu = cp.random.uniform(-1.0, 1.0, n_samples, dtype=cp.float32)
        dsigma_domega_gpu = (alpha**2 / (4.0 * s)) * (1.0 + cos_theta_gpu**2)
        sigma_gpu_pb = 4.0 * cp.pi * cp.mean(dsigma_domega_gpu) * GEV2_TO_PB
        sigma_gpu_unc_pb = (
            4.0
            * cp.pi
            * cp.std(dsigma_domega_gpu)
            / cp.sqrt(cp.float32(n_samples))
            * GEV2_TO_PB
        )

        end_event.record()
        end_event.synchronize()
        gpu_time_ms = cp.cuda.get_elapsed_time(start_event, end_event)

        sigma_gpu_pb = float(sigma_gpu_pb)
        sigma_gpu_unc_pb = float(sigma_gpu_unc_pb)

        print(f"- Result: {sigma_gpu_pb:.6f} ± {sigma_gpu_unc_pb:.6f} pb")
        print(f"- Time:   {gpu_time_ms:.2f} ms")
        print(f"- Pull vs theory: {(sigma_gpu_pb - sigma_analytic_pb) / sigma_gpu_unc_pb:.2f} sigma")
        print(f"- Speedup vs CPU: {cpu_time_ms / gpu_time_ms:.1f}x")
    else:
        print("\nGPU path skipped because CuPy is not available")


exercise_solution_cross_section()

---
## 9. Summary and Key Takeaways

### What the upstream tools actually do

- **MadGraph5_aMC@NLO**: hard-scattering matrix elements and parton-level event generation, usually via CLI workflows
- **PYTHIA 8**: showers, hadronization, and many decays in a C++ event-generator framework
- **EvtGen**: dedicated heavy-flavor decay modeling, typically inside larger C++ software stacks

### Where GPU acceleration fits today

- Parallel matrix-element scans and Monte Carlo integration are natural GPU workloads
- Post-generation analysis of large event arrays is often easier to accelerate than the generator itself
- Toy decay kernels are useful stand-ins for understanding the arithmetic intensity of decay studies

### Practical lessons

1. Keep the physics story consistent: separate hard-process, shower, and heavy-flavor examples when they are not one real chain.
2. Be explicit about upstream interfaces: command-line or C++ tools should not be advertised as turnkey Python GPU libraries.
3. Validate the normalization and units carefully. The GeV^-2 to pb conversion matters.
4. Use the GPU where the work is truly data parallel, and keep the notebook honest about the rest.

### Natural next steps

- Replace the toy MadGraph-style sample with real LHE events produced externally by MG5_aMC.
- Read a real HepMC or ROOT output sample from a shower generator and run the same GPU analysis patterns on it.
- Extend the decay study to sequential decays or acceptance cuts matched to a specific detector.